In [ ]:
import os
import ROOT
import subprocess
from ROOT import std

# 1. Setup Paths
base_path = "/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod"
# Let's get the ROOT lib directory dynamically to ensure it's correct
try:
    root_lib_dir = subprocess.check_output(['root-config', '--libdir'], text=True).strip()
except:
    root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"

# 2. Update Environment
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gSystem.AddDynamicPath(root_lib_dir)

# 3. Load the "Big" ROOT dependency blocks
# Loading these covers almost all physics class requirements (Hist, Geom, Graf, etc.)
root_libs = ["libRint","libROOTGpadv7","libMathMore","libCore", "libRIO", "libNet", "libHist", "libGraf", "libGraf3d", "libGpad", "libTree", "libMathCore", "libThread", "libMatrix", "libGeom","libROOTHist"]
for lib in root_libs:
    ROOT.gSystem.Load(lib)

def load_custom_class(class_name, extension="cpp"):
    source_file = os.path.join(base_path, f"{class_name}.{extension}")
    so_file = os.path.join(base_path, f"{class_name}_{extension}.so")
    
    # Try to load existing
    if os.path.exists(so_file):
        # We check return status: 0 = loaded, 1 = already loaded, -1 = failure
        if ROOT.gSystem.Load(so_file) >= 0:
            print(f"📦 Loaded: {class_name}")
            return True
    
    # If load failed or file missing, compile
    print(f"🛠️  Compiling: {class_name}...")
    return ROOT.gSystem.CompileMacro(source_file, "kO") >= 0

# 4. Run the sequence
if load_custom_class("PhysdEdx"):
    if load_custom_class("Hypfit"):
        from ROOT import Hypfit
        h_fit = Hypfit()
        print("🚀 Success! All libraries and dependencies are active.")

In [ ]:
'''
import os
import ROOT

# 1. Environment & Paths
root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gInterpreter.AddIncludePath("/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod/")

# 2. Compile PhysdEdx first and keep it in memory
# The 'k' means keep the shared library, 'O' is optimize, 'f' is force
print("Step 1: Compiling PhysdEdx...")
ROOT.gSystem.CompileMacro("/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod/PhysdEdx.cpp", "kOf")

# 3. Explicitly load the newly created shared library for PhysdEdx
# ACLiC usually names it [filename]_cxx.so
phys_so = "/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod/PhysdEdx_cpp.so"
ROOT.gSystem.Load(phys_so)

# 4. Compile Hypfit while linked to PhysdEdx
print("Step 2: Compiling Hypfit...")
ROOT.gSystem.CompileMacro("/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod/Hypfit.cpp", "kOf")

# 5. Import and Instantiate
try:
    from ROOT import Hypfit
    h_fit = Hypfit()
    print("🚀 Success! Hypfit is ready.")
except Exception as e:
    print("❌ Error:", e)
'''

In [ ]:
print(ROOT.gSystem.GetDynamicPath())

In [ ]:

# Step 4: Prepare vectors
this_rr_vec    = std.vector('double')()
this_dEdx_vec  = std.vector('double')()
this_pitch_vec = std.vector('double')()

rr_list    = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
dEdx_list  = [2.1, 3.5, 1.7, 2.0, 3.0, 2.5, 0.5]
pitch_list = [0.5, 0.7, 0.6, 0.3, 0.2, 0.3, 0.3]

for x in rr_list:
    this_rr_vec.push_back(x)
for x in dEdx_list:
    this_dEdx_vec.push_back(x)
for x in pitch_list:
    this_pitch_vec.push_back(x)

# Step 5: Call GetTLExtensionP
target_PDG = 211
best_plane = 0
cleaning_method = "none"

reco_P = h_fit.GetTLExtensionP(
    target_PDG,
    this_rr_vec,
    this_dEdx_vec,
    this_pitch_vec,
    best_plane,
    cleaning_method
)

print("Reconstructed momentum [GeV]:", reco_P)